In [1]:
import csv
import spacy
import scispacy
from scispacy.linking import EntityLinker
from scispacy.abbreviation import AbbreviationDetector

In [2]:
nlp = spacy.load("en_core_sci_sm")
nlp.add_pipe("abbreviation_detector")
nlp.add_pipe("scispacy_linker",
    config={
        "linker_name": "umls",
        "resolve_abbreviations": True,
        "threshold": 0.80,
        "max_entities_per_mention": 5,
    },
)

c:\Users\Laptop\Personal-Health-Passport\.venv\Lib\site-packages\spacy\language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]
c:\Users\Laptop\Personal-Health-Passport\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.1.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Laptop\Personal-Health-Passport\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.1.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/mod

In [3]:
text = """There is evidence of cancer."""

text2 = """
History:
The patient has lupus nephritis and hypertension. She was started on mycophenolate mofetil (MMF) and prednisone six months ago.

Assessment:
Proteinuria has significantly improved since treatment began. Renal function remains stable. The patient's fatigue has decreased, but the joint pain has worsened over the past two weeks. Blood pressure is well controlled. 
Serum creatinine has increased slightly compared with the previous visit. The rash has completely resolved. 
The patient denies fever, chest pain, and shortness of breath. Mild nausea developed after increasing the MMF dose but has since subsided.

Plan:
Continue MMF and prednisone. Reduce the steroid dose gradually. Repeat renal function tests in four weeks.
"""

text3 = """ The patient's cough has improved, although the dyspnoea continues to worsen. 
Peripheral oedema has almost completely resolved, while serum creatinine remains elevated. 
There is no evidence of infection, and the patient denies fever. 
Methotrexate was discontinued because liver enzymes increased significantly. 
Overall, rheumatoid arthritis appears to be responding well to treatment.
"""

text4 = """
Assessment

The patient has biopsy-proven lupus nephritis and is currently receiving mycophenolate mofetil and rituximab.

Overall, rheumatoid arthritis appears to be responding well to treatment. Proteinuria has significantly improved since the previous visit, although renal function remains stable. Serum creatinine is unchanged.

Dyspnoea continues to worsen despite corticosteroid therapy, and mild ankle oedema has developed over the past week.

There is no evidence of active infection, and the patient denies fever, chest pain, shortness of breath, abdominal pain, or haematuria.

The skin rash has completely resolved, but intermittent joint stiffness persists. Fatigue remains mildly improved following treatment.

Blood pressure is well controlled on the current medication regimen.

Plan

Continue mycophenolate mofetil. 
Reduce the prednisolone dose if renal function remains stable. 
Repeat renal function tests and urine protein-to-creatinine ratio in four weeks. 
Monitor dyspnoea closely and consider chest CT if symptoms continue to worsen.
"""

doc = nlp(text)
doc2 = nlp(text2)
doc3 = nlp(text3)
doc4 = nlp(text4)

c:\Users\Laptop\Personal-Health-Passport\.venv\Lib\site-packages\scispacy\abbreviation.py:248: UserWarning: [W036] The component 'matcher' does not have any patterns defined.
  global_matches = self.global_matcher(doc)


In [4]:
semantic_types = {}

with open("semantic_types.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        semantic_types[row["code"]] = row

In [5]:
def get_semantic_class(code):
    info = semantic_types.get(code)

    if info is None:
        return "Unknown"
    
    return info["category"]

In [6]:
allowed_semantic_codes = {
    # Disorders, findings and symptoms
    "T019", "T020", "T033", "T037",
    "T046", "T047", "T048", "T049",
    "T184", "T190", "T191",

    # Laboratory or test results
    "T034",
    
    # Clinical attributes/functions
    "T032", "T039", "T040", "T042", "T201",

    # Medication
    "T121", "T195", "T200",
    
    # Procedures
    "T059", "T060", "T061"
}

In [7]:
improvement_words = {
    "improve",
    "decrease",
    "recover",
    "remit",
    "respond"
}

resolution_words = {
    "resolve",
    "subside"
}

worsening_words = {
    "worse",
    "worsen",
    "deteriorate",
    "decline",
    "progress",
    "exacerbate",
    "increase",
    "rise",
    "relapse"
}

stable_words = {
    "stable",
    "remain",
    "unchanged",
    "control",
    "persist"
}

new_words = {
    "develop",
    "arise",
    "emerge"
}

existence_words = {
    "have",
    "diagnose"
}

medication_action_words = {
    "start": "started",
    "continue": "continued",
    "stop": "stopped",
    "discontinue": "stopped",
    "hold": "held"
}

dose_action_words = {
    "reduce": "dose_reduced",
    "increase": "dose_increased"
}

In [8]:
negation_lemmas = {
    "deny",
    "exclude"
}

negation_words = {
    "no",
    "not",
    "without",
    "neither"
}

def check_negation(token):
    nodes = [token, *list(token.ancestors)]

    for node in nodes:
        if node.lemma_.lower() in negation_lemmas:
            return True

        for child in node.children:
            if (child.dep_ == "neg" or child.text.lower() in negation_words):
                return True

    return False

In [9]:
status_entity_lemmas = (
    improvement_words
    | resolution_words
    | worsening_words
    | stable_words
    | new_words
    | existence_words
    | negation_lemmas
    | set(medication_action_words)
    | set(dose_action_words)
)

note_headings = {
    "history",
    "assessment",
    "impression",
    "plan",
}

entiy_discourse_words = {
    "overall",
    "generally",
    "clinically"
}

In [10]:
def get_entity_info(doc, minimum_score=0.80):
    entities = []
    linker = nlp.get_pipe("scispacy_linker")

    abbreviation_map = {
        str(abbr): str(abbr._.long_form)
        for abbr in doc._.abbreviations
    }

    print("Abbreviation Map:", abbreviation_map)

    for entity in doc.ents:
        entity_text = entity.text

        entity_lemmas = {
            token.lemma_.lower()
            for token in entity
            if not token.is_punct and not token.is_space
        }

        if (entity_lemmas and entity_lemmas.issubset(status_entity_lemmas)):
            continue

        if (entity_text.strip().lower() in note_headings):
            continue

        normalised_text = abbreviation_map.get(entity_text, entity_text)

        selected_candidate = None

        for cui, score in entity._.kb_ents:
            if score < minimum_score:
                continue

            concept = linker.kb.cui_to_entity[cui]

            allowed_codes = [code for code in concept.types if code in allowed_semantic_codes]

            if not allowed_codes:
                continue

            selected_candidate = {
                "cui": cui,
                "score": float(score),
                "concept": concept,
                "semantic_codes": allowed_codes,
            }
            
            break

        if selected_candidate is None:
            continue

        concept = selected_candidate["concept"]

        entities.append({
            "entity": entity,
            "text": entity_text,
            "tokens": list(entity),
            "normalised": normalised_text,
            "canonical": concept.canonical_name,
            "cui": selected_candidate["cui"],
            "semantic_codes": selected_candidate["semantic_codes"],
            "semantic_types": [get_semantic_class(code) for code in selected_candidate["semantic_codes"]],
            "score": selected_candidate["score"],
            "start": entity.start_char,
            "end": entity.end_char,
        })

    return entities

for entity in get_entity_info(doc3):
    print(entity)

Abbreviation Map: {}
{'entity': dyspnoea, 'text': 'dyspnoea', 'tokens': [dyspnoea], 'normalised': 'dyspnoea', 'canonical': 'Dyspnea', 'cui': 'C0013404', 'semantic_codes': ['T184'], 'semantic_types': ['Disorders'], 'score': 0.9848591685295105, 'start': 48, 'end': 56}
{'entity': oedema, 'text': 'oedema', 'tokens': [oedema], 'normalised': 'oedema', 'canonical': 'Edema', 'cui': 'C0013604', 'semantic_codes': ['T046'], 'semantic_types': ['Disorders'], 'score': 0.9804628491401672, 'start': 90, 'end': 96}
{'entity': serum creatinine, 'text': 'serum creatinine', 'tokens': [serum, creatinine], 'normalised': 'serum creatinine', 'canonical': 'Creatinine measurement, serum (procedure)', 'cui': 'C0201976', 'semantic_codes': ['T059'], 'semantic_types': ['Procedures'], 'score': 0.9884899854660034, 'start': 135, 'end': 151}
{'entity': elevated, 'text': 'elevated', 'tokens': [elevated], 'normalised': 'elevated', 'canonical': 'ST segment elevation (finding)', 'cui': 'C0520886', 'semantic_codes': ['T033']

In [11]:
def get_conj_entities(token):
    entities = [token]

    for child in token.children:
        if child.dep_ == "conj":
            # entities.append(child)
            entities.extend(get_conj_entities(child))

    return entities

In [12]:
def build_entity_phrase(token):
    included_tokens = {token.i: token}

    for child in token.children:
        if (child.dep_ in {"amod", "compound", "nmod"} and child.lemma_.lower() not in entiy_discourse_words):
            for subtree_token in child.subtree:
                included_tokens[subtree_token.i] = subtree_token

    tokens = [included_tokens[index] for index in sorted(included_tokens)]

    return " ".join(
        item.text
        for item in tokens
        if not item.is_space and not item.is_punct and item.dep_ != "det" 
        and item.lemma_.lower() not in entiy_discourse_words
    )

In [13]:
def print_tokens(doc):
    for token in doc:
        print(
            f"{token.i:<15}"
            f"{token.text:<15}"
            f"POS={token.pos_:<6}"
            f"DEP={token.dep_:<12}"
            f"HEAD={token.head.text}"
        )

print_tokens(doc2)

0              
              POS=SPACE DEP=punct       HEAD=History
1              History        POS=NOUN  DEP=nsubj       HEAD=has
2              :              POS=PUNCT DEP=punct       HEAD=has
3              
              POS=SPACE DEP=punct       HEAD=has
4              The            POS=DET   DEP=det         HEAD=patient
5              patient        POS=NOUN  DEP=nsubj       HEAD=has
6              has            POS=AUX   DEP=ROOT        HEAD=has
7              lupus          POS=NOUN  DEP=compound    HEAD=nephritis
8              nephritis      POS=NOUN  DEP=dobj        HEAD=has
9              and            POS=CCONJ DEP=cc          HEAD=nephritis
10             hypertension   POS=NOUN  DEP=conj        HEAD=nephritis
11             .              POS=PUNCT DEP=punct       HEAD=has
12             She            POS=PRON  DEP=nsubjpass   HEAD=started
13             was            POS=AUX   DEP=auxpass     HEAD=started
14             started        POS=VERB  DEP=ROOT        

In [14]:
def classify_trigger(token):
    lemma = token.lemma_.lower()

    result = {
        "assertion": "present",
        "trend": None,
        "action": None,
    }

    if lemma in negation_lemmas:
        result["assertion"] = "absent"

    elif lemma in improvement_words:
        result["trend"] = "improving"

    elif lemma in resolution_words:
        result["assertion"] = "absent"
        result["trend"] = "resolved"

    elif lemma in worsening_words:
        result["trend"] = "worsening"

    elif lemma in stable_words:
        result["trend"] = "stable"

        child_lemmas = {child.lemma_.lower() for child in token.children}

        if "elevated" in child_lemmas:
            result["trend"] = "stable_abnormal"

        elif "improved" in child_lemmas:
            result["trend"] = "improving"

    elif lemma in new_words:
        result["trend"] = "new"

    if lemma in medication_action_words:
        result["action"] = (medication_action_words[lemma])

    return result

In [15]:
def build_trigger_phrase(token):
    included_tokens = [token]

    for child in token.children:
        if child.dep_ in {"advmod", "neg", "xcomp", "acomp", "attr", "oprd"}:
            included_tokens.append(child)

    included_tokens.sort(key=lambda item: item.i)

    return " ".join(item.text for item in included_tokens)

In [16]:
def extract_no_evidence_relationships(doc):
    relationships = []

    for evidence in doc:
        if evidence.lemma_.lower() != "evidence":
            continue

        has_no = any(
            child.text.lower() == "no"
            for child in evidence.children
        )

        if not has_no:
            continue

        for child in evidence.children:
            if child.dep_ in {
                "nmod",
                "pobj",
            }:
                relationships.append({
                    "_token": child,
                    "entity": build_entity_phrase(child),
                    "trigger": "no evidence",
                    "assertion": "absent",
                    "trend": None,
                    "action": None,
                })

    return relationships

In [17]:
def extract_clinical_relationships(doc):
    trigger_lemmas = (
        improvement_words
        | resolution_words
        | worsening_words
        | stable_words
        | new_words
        | existence_words
        | negation_lemmas
        | set(medication_action_words)
        | set(dose_action_words)
    )

    relationships = []
    seen = set()

    for trigger in doc:
        lemma = trigger.lemma_.lower()

        if lemma not in trigger_lemmas:
            continue

        if trigger.dep_ in {"xcomp", "acomp", "attr", "oprd"}:
            for ancestor in trigger.ancestors:
                subjects = [
                    child
                    for child in ancestor.children
                    if child.dep_ in {"nsubj", "nsubjpass"} and child.pos_ in {"NOUN", "PROPN"}
                ]

                if subjects:
                    for subject in subjects:
                        argument_tokens.extend(get_conj_entities(subject))

                    break

        context = classify_trigger(trigger)
        argument_tokens = []

        if lemma in negation_lemmas:
            allowed_dependencies = {"dobj", "obj", "attr", "conj"}

        elif lemma in medication_action_words:
            allowed_dependencies = {"dobj", "obj", "attr", "conj", "nsubjpass", "nmod", "pobj"}
        
        elif lemma in existence_words:
            allowed_dependencies = {"dobj", "obj", "attr", "conj", "nmod"}

        else:
            allowed_dependencies = {"dobj", "obj", "conj", "nsubj", "nsubjpass"}

        for child in trigger.children:
            if (child.dep_ in allowed_dependencies and child.pos_ in {"NOUN", "PROPN"}):
                argument_tokens.extend(get_conj_entities(child))

        if trigger.dep_ in {"xcomp", "acomp", "attr", "oprd"}:
            governing_verb = trigger.head

            for child in governing_verb.children:
                if child.dep_ in {"nsubj", "nsubjpass"}:
                    argument_tokens.extend(get_conj_entities(child))

        if (not argument_tokens and trigger.dep_ == "conj"):
            governing_trigger = trigger.head

            for child in governing_trigger.children:
                if child.dep_ in {"nsubj", "nsubjpass", "dobj", "obj"}:
                    argument_tokens.extend(get_conj_entities(child))

        argument_tokens = list({token.i: token for token in argument_tokens}.values())

        for argument in argument_tokens:
            entity_phrase = build_entity_phrase(argument)

            assertion = context["assertion"]

            if check_negation(argument):
                assertion = "absent"

            action = context["action"]
            trend = context["trend"]

            entity_words = {word.lower() for word in entity_phrase.split()}

            if lemma in dose_action_words:
                if "dose" in entity_words:
                    action = dose_action_words[lemma]
                    trend = None
                else:
                    action = None

            result = {
                "_token": argument,
                "entity": entity_phrase,
                "trigger": build_trigger_phrase(trigger),
                "assertion": assertion,
                "trend": trend,
                "action": action,
            }

            result_key = (
                result["entity"].lower(),
                result["trigger"].lower(),
                result["assertion"],
                result["trend"],
                result["action"],
            )

            if result_key not in seen:
                relationships.append(result)
                seen.add(result_key)
                
            for result in extract_no_evidence_relationships(doc):
                result_key = (
                    result["entity"].lower(),
                    result["trigger"].lower(),
                    result["assertion"],
                    result["trend"],
                    result["action"],
                )

                if result_key not in seen:
                    relationships.append(result)
                    seen.add(result_key)

    return relationships
        
for relationship in extract_clinical_relationships(doc4):
    print(relationship)


{'_token': nephritis, 'entity': 'biopsy-proven lupus nephritis', 'trigger': 'has', 'assertion': 'present', 'trend': None, 'action': None}
{'_token': infection, 'entity': 'active infection', 'trigger': 'no evidence', 'assertion': 'absent', 'trend': None, 'action': None}
{'_token': arthritis, 'entity': 'rheumatoid arthritis', 'trigger': 'responding well', 'assertion': 'present', 'trend': 'improving', 'action': None}
{'_token': Proteinuria, 'entity': 'Proteinuria', 'trigger': 'significantly improved', 'assertion': 'present', 'trend': 'improving', 'action': None}
{'_token': function, 'entity': 'renal function', 'trigger': 'remains stable', 'assertion': 'present', 'trend': 'stable', 'action': None}
{'_token': function, 'entity': 'renal function', 'trigger': 'stable', 'assertion': 'present', 'trend': 'stable', 'action': None}
{'_token': creatinine, 'entity': 'Serum creatinine', 'trigger': 'unchanged', 'assertion': 'present', 'trend': 'stable', 'action': None}
{'_token': Dyspnoea, 'entity': '

In [18]:
def find_umls_entity_for_token(token, linked_entities, relationship_text=None):
    candidates = []

    for linked_entity in linked_entities:
        entity_span = linked_entity["entity"]

        if entity_span.start <= token.i < entity_span.end:
            candidates.append(linked_entity)

    if not candidates:
        return None

    candidates.sort(key=lambda entity: (len(entity["entity"]), entity["score"]), reverse=True)

    best = candidates[0]

    if relationship_text is not None:
        relationship_text_lower = relationship_text.lower()
        linked_text_lower = best["text"].lower()

        if linked_text_lower not in relationship_text_lower:
            return None

        is_partial_match = (linked_text_lower != relationship_text_lower)

        if (is_partial_match and best["score"] < 0.90):
            return None

    return best

In [19]:
def find_umls_entity_by_text(relationship_text, linked_entities):
    search_text = relationship_text.strip().lower()

    matches = [
        linked_entity
        for linked_entity in linked_entities
        if linked_entity["text"].strip().lower() == search_text
    ]

    if not matches:
        return None

    return max(matches, key=lambda entity: entity["score"])

In [20]:
def extract_linked_relationships(doc):
    relationships = extract_clinical_relationships(doc)
    linked_entities = get_entity_info(doc)

    results = []

    for relationship in relationships:
        token = relationship.get("_token")

        linked = find_umls_entity_for_token(token, linked_entities, relationship["entity"])

        if linked is None:
            linked = find_umls_entity_by_text(relationship["entity"], linked_entities)

        result = {
            key: value 
            for key, value in relationship.items() 
            if key != "_token"
        }

        result["cui"] = (linked["cui"] if linked else None)
        result["canonical"] = (linked["canonical"] if linked else None)
        result["semantic_codes"] = (linked["semantic_codes"] if linked else [])

        results.append(result)

    return results

In [21]:
for relationship in extract_linked_relationships(doc4):
    print(relationship)

Abbreviation Map: {}
{'entity': 'biopsy-proven lupus nephritis', 'trigger': 'has', 'assertion': 'present', 'trend': None, 'action': None, 'cui': None, 'canonical': None, 'semantic_codes': []}
{'entity': 'active infection', 'trigger': 'no evidence', 'assertion': 'absent', 'trend': None, 'action': None, 'cui': None, 'canonical': None, 'semantic_codes': []}
{'entity': 'rheumatoid arthritis', 'trigger': 'responding well', 'assertion': 'present', 'trend': 'improving', 'action': None, 'cui': 'C0003873', 'canonical': 'Rheumatoid Arthritis', 'semantic_codes': ['T047']}
{'entity': 'Proteinuria', 'trigger': 'significantly improved', 'assertion': 'present', 'trend': 'improving', 'action': None, 'cui': 'C0033687', 'canonical': 'Proteinuria', 'semantic_codes': ['T033']}
{'entity': 'renal function', 'trigger': 'remains stable', 'assertion': 'present', 'trend': 'stable', 'action': None, 'cui': 'C0232804', 'canonical': 'Renal function', 'semantic_codes': ['T042']}
{'entity': 'renal function', 'trigger